In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix


Using device: cuda


In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Dataset class
class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.label_map = {
            "normal": 0,
            "bacterial": 1,
            "viral": 2
        }

        for label_name in os.listdir(root_dir):
            full_path = os.path.join(root_dir, label_name)
            if os.path.isdir(full_path) and label_name in self.label_map:
                for file in os.listdir(full_path):
                    self.samples.append((os.path.join(full_path, file), self.label_map[label_name]))
            else:
                print(f"Skipping unknown or invalid class: {label_name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            print(f"Error loading image: {image_path}")
            image = Image.new("RGB", (224, 224))

        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# Transforms
image_size = 224
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Datasets and loaders
train_dataset = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained", transform)
val_dataset   = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed", transform)
test_dataset  = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested", transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, num_workers=2)


In [ ]:
# CNN + Transformer Hybrid Model (ResFormer)
class ResFormer(nn.Module):
    def __init__(self, num_classes=3):
        super(ResFormer, self).__init__()
        base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(base_model.children())[:-2])  # Output: [B, 2048, 7, 7]
        self.flatten = nn.Flatten(2)  # [B, 2048, 49]
        self.transpose = lambda x: x.permute(0, 2, 1)  # [B, 49, 2048]

        #  Transformer Encoder block
        encoder_layer = nn.TransformerEncoderLayer(d_model=2048, nhead=8, dim_feedforward=4096, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # Classification Head
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.cnn(x)                # -> [B, 2048, 7, 7]
        x = self.flatten(x)            # -> [B, 2048, 49]
        x = self.transpose(x)          # -> [B, 49, 2048]
        x = self.transformer(x)        # -> [B, 49, 2048]
        x = x.mean(dim=1)              # Global average over sequence
        return self.classifier(x)



In [ ]:
#  Train function
def train_model(model, epochs=5):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            if batch_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx}], Loss: {loss.item():.4f}")

        acc = 100 * correct / total
        val_acc = validate_model(model)
        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {acc:.2f}%, Val Acc: {val_acc:.2f}%")

In [ ]:
#  Validation function
def validate_model(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    model.train()
    return 100 * correct / total

In [ ]:
#  Evaluation function
def evaluate_model(model):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["normal", "bacterial", "viral"]))


In [ ]:
# Train + Evaluate
model = ResFormer().to(device)
train_model(model, epochs=5)

In [ ]:
print("\nTesting Accuracy:")
evaluate_model(model)